In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
from sklearn.model_selection import train_test_split

In [10]:
import json

# Investigating data

In [3]:
df = pd.read_csv("../data/balanced_ai_human_prompts.csv")

In [58]:
(df['generated'] == 0).sum()

np.int64(1375)

In [59]:
(df['generated'] == 1).sum()

np.int64(1375)

=> equal number of data for both sides 

In [60]:
df.tail()

,text,generated
2745,Generate a detailed summary of global healthca...,1
2746,Compose an in-depth exploration of financial t...,1
2747,Generate a detailed summary of autonomous vehi...,1
2748,Develop a persuasive argument about internet o...,1
2749,Generate a detailed summary of supply chain ma...,1


In [4]:
df = df.sample(frac=1).reset_index(drop=True)
df.tail()

,text,generated
2745,Transportation is a large necessity in most co...,0
2746,Write a comprehensive essay explaining interne...,1
2747,The age of cars has come to a grinding stop. A...,0
2748,Dear Senator the electoral voting system shoul...,0
2749,"From riding horses with wagons to, driving car...",0


In [62]:
df['text'].str.len().quantile(0.95)

np.float64(4404.749999999998)

In [5]:
# dividng train test
X = np.array(df['text'].values)
y = np.array(df['generated'].values)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

X_train: (2200,)
X_test: (550,)
y_train: (2200,)
y_test: (550,)


# Tokenize

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [7]:
vocab_size = 16000
max_len = 4405
embedding_dim = 128

In [8]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [9]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')

X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

In [11]:
tokenizer_json = tokenizer.to_json()
with open('tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))

# Train

In [ ]:
model = keras.Sequential([
    keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_len,
        mask_zero=True 
    ),
    keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(32)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [69]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [70]:
history = model.fit(
    X_train_pad, y_train,
    batch_size=32,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 18s 300ms/step - accuracy: 0.8381 - loss: 7.1885 - val_accuracy: 0.9977 - val_loss: 0.0189
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 17s 306ms/step - accuracy: 0.9864 - loss: 0.1797 - val_accuracy: 0.9909 - val_loss: 0.0648
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 343ms/step - accuracy: 0.9955 - loss: 0.0308 - val_accuracy: 0.9977 - val_loss: 0.0111
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 337ms/step - accuracy: 0.9949 - loss: 0.0113 - val_accuracy: 0.9977 - val_loss: 0.0045
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 18s 335ms/step - accuracy: 0.9949 - loss: 0.0100 - val_accuracy: 0.9977 - val_loss: 0.0036
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 339ms/step - accuracy: 0.9972 - loss: 0.0059 - val_accuracy: 0.9977 - val_loss: 0.0050
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 339ms/step - accuracy: 0.9983 - loss: 0.0113 - val_accuracy: 0.9977 - val_loss: 0.0051
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 337ms/step - accuracy: 0.9983 - loss: 0.0028 - val_accu

In [71]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

Test Accuracy: 0.9709


In [ ]:
model.save('alp.h5')

# Predictions

In [ ]:
loaded_model = keras.models.load_model('alp.h5')

In [74]:
text_sample = [
    'It is so so sunny today. I should put on some sunscreen.',
    'Access to quality education has long been regarded as a fundamental human right, yet millions of children around the world remain excluded from meaningful learning opportunities. In the twenty-first century, this exclusion has taken on a new dimension: the digital divide. As classrooms increasingly rely on technology — from online learning platforms to digital textbooks — students without reliable internet access or devices are being left further behind. While wealthier schools equip students with the tools needed to thrive in a technology-driven world, those in low-income or rural communities struggle to keep pace. Bridging this gap is no longer a matter of convenience but of equity, and without deliberate action, digital inequality threatens to deepen the very educational disparities that universal education policies seek to eliminate.'
]
input_sample = np.array(text_sample)
input_sample = tokenizer.texts_to_sequences(input_sample)
input_sample = pad_sequences(input_sample, maxlen=max_len, padding='post', truncating='post')

In [75]:
predictions = loaded_model.predict(input_sample)
class_labels = (predictions > 0.5).astype(int)
class_labels, predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step


(array([[1],
        [1]]),
 array([[1.],
        [1.]], dtype=float32))